# Story 1.2: Simulation Graph E2E Test

This notebook tests the LangGraph simulation workflow end-to-end with real AI agents.

**What this tests:**
1. Complete scenario generation
2. LangGraph workflow with interrupt() for human-in-the-loop
3. Full simulation cycle through multiple turns
4. MLflow tracing of multi-step execution

## 1. Setup & Imports

In [1]:
import warnings

# Suppress MLflow/LangGraph autologging warnings
warnings.filterwarnings("ignore", message=".*autologging.*")
warnings.filterwarnings("ignore", message=".*LangChain.*")

from summit_sim.agents.generator import generate_scenario
from summit_sim.graphs.simulation import create_simulation_graph
from summit_sim.schemas import HostConfig, generate_scenario_id
from summit_sim.settings import settings
from summit_sim.tracing import (
    enable_tracing,
    log_simulation_results,
    simulation_session,
)

# Initialize MLflow with unified tracing for Pydantic AI and LangGraph
enable_tracing()

print(f"✓ MLflow tracking URI: {settings.mlflow_tracking_uri}")
print(f"✓ Experiment name: {settings.mlflow_experiment_name}")
print(f"✓ Model: {settings.default_model}")
print(f"✓ API Key configured: {bool(settings.openrouter_api_key)}")
print("\n📊 MLflow Tracing Architecture:")
print("  • Scenario generation → Independent Pydantic AI traces")
print("  • Simulation loop → Nested under parent run with session_id")
print("  • class_id → Links generation and simulation traces in MLflow")
print("  • Interrupt 'exceptions' → Expected (human-in-the-loop mechanism)")

✓ MLflow tracking URI: https://mlflow.bhamm-lab.com
✓ Experiment name: summit-sim
✓ Model: google/gemini-3.1-flash-lite-preview
✓ API Key configured: True

📊 MLflow Tracing Architecture:
  • Scenario generation → Independent Pydantic AI traces
  • Simulation loop → Nested under parent run with session_id
  • class_id → Links generation and simulation traces in MLflow
  • Interrupt 'exceptions' → Expected (human-in-the-loop mechanism)


## 2. Generate Scenario

First, we'll generate a complete scenario with multiple turns.

**Note**: Scenario generation creates an independent Pydantic AI trace in MLflow, separate from the simulation session. This is by design - generation and simulation are distinct phases.

In [2]:
# Test configuration - class_id auto-generated if not provided
config = HostConfig(num_participants=3, activity_type="hiking", difficulty="med")

print(f"Generating scenario: {config}")
print(f"📋 class_id: {config.class_id}")
print("-" * 60)

# Generate scenario
scenario = await generate_scenario(config)

# Generate unique scenario ID
scenario_id = generate_scenario_id()

print(f"✓ Title: {scenario.title}")
print(f"✓ Setting: {scenario.setting}")
print(f"✓ Patient: {scenario.patient_summary}")
print(f"✓ Turns: {len(scenario.turns)}")
print(f"✓ Starting turn: {scenario.starting_turn_id}")
print("\nLearning Objectives:")
for obj in scenario.learning_objectives:
    print(f"  • {obj}")

print("\n💡 In MLflow UI, filter by class_id to see all related traces")

Generating scenario: num_participants=3 activity_type='hiking' difficulty='med' class_id=None
📋 class_id: None
------------------------------------------------------------
✓ Title: Alpine Misdiagnosis
✓ Setting: A remote Alpine hiking trail, 4 miles from the trailhead. Weather is turning cold and overcast, approximately 50°F (10°C).
✓ Patient: 42-year-old male hiker, alert and oriented. Complaining of right-sided lower abdominal pain after a minor trip 2 hours ago. Pale, slightly tachycardic.
✓ Turns: 4
✓ Starting turn: 0

Learning Objectives:
  • Perform a thorough secondary assessment, including a focused abdominal exam.
  • Distinguish between musculoskeletal injury and acute visceral pathology in a wilderness setting.
  • Prioritize urgent evacuation based on clinical suspicion of a surgical emergency.

💡 In MLflow UI, filter by class_id to see all related traces


Trace(trace_id=tr-00520405394fd932021dd563ecadb00e)

## 3. Display Scenario Turns

Let's see what turns were generated:

In [3]:
for _i, turn in enumerate(scenario.turns):
    print()
    print("=" * 60)
    marker = "(START)" if turn.is_starting_turn else ""
    print(f"Turn {turn.turn_id} {marker}")
    print("=" * 60)
    print()
    print(turn.narrative_text)
    print()
    print("Choices:")
    for choice in turn.choices:
        if choice.next_turn_id is None:
            end = "END"
        else:
            end = f"Turn {choice.next_turn_id}"
        mark = "✓" if choice.is_correct else " "
        print(f"  [{mark}] {choice.description} -> {end}")


Turn 0 (START)

You encounter a group of three hikers. One man is sitting on a rock, hunched over, clutching his right side. His companions say he tripped 2 hours ago. He tells you his side hurts a lot, and he feels nauseous. What is your first priority?

Choices:
  [ ] Perform a quick physical check of his ankles and knees, then encourage him to walk slowly to the trailhead to get warm. -> Turn 1
  [✓] Conduct a full vital sign check and a focused abdominal exam (asking about pain onset, location, and performing light palpation). -> Turn 1

Turn 1 

You perform the assessment. The patient is pale and tachycardic. He denies a direct blow to the abdomen. Upon light palpation, he guards the Right Lower Quadrant (RLQ). He explicitly winces when you lift your hand quickly after pressing on that area. How do you interpret these findings?

Choices:
  [ ] Assume a pulled muscle from the fall, administer over-the-counter painkillers, and advise a slow walk out. -> Turn 2
  [✓] Note his elevat

## 4. Initialize Simulation Graph

Create the LangGraph workflow with checkpointer for interrupt support:

In [4]:
# Create the graph with in-memory checkpointer
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()
graph = create_simulation_graph(checkpointer=memory)

# Initialize state (TypedDict - use dict syntax)
initial_state = {
    "scenario_draft": scenario,
    "current_turn_id": scenario.starting_turn_id,
    "transcript": [],
    "is_complete": False,
    "key_learning_moments": [],
    "last_selected_choice": None,
    "simulation_result": None,
    "scenario_id": scenario_id,
    "class_id": config.class_id,
}

print("✓ Graph created successfully")
print(f"✓ Initial state: turn_id={initial_state['current_turn_id']}")
print(f"✓ class_id: {initial_state['class_id']}")
print("\n⚠️  Next cell will start MLflow session with parent run tracking")

✓ Graph created successfully
✓ Initial state: turn_id=0
✓ class_id: None

⚠️  Next cell will start MLflow session with parent run tracking


## 5. Run Simulation with MLflow Session Tracking

Execute the full simulation workflow within an MLflow session. The simulation will:
1. Create a parent MLflow run with session metadata
2. Present each turn (with traces nested under parent)
3. Pause with `interrupt()` for your choice
4. Call the AI agent for feedback
5. Advance to next turn
6. Log final results to the parent run

**MLflow Features:**
- Session ID links all traces together
- Parent run contains all agent calls as child spans
- Automatic error tracking (status: completed/failed)
- Final metrics: total_turns, is_complete, learning_moments_count

**⚠️ Expected Behavior:**
- `interrupt()` raises exceptions for human-in-the-loop - these appear in MLflow traces but are NOT errors
- Scenario generation (Cell 2) creates separate Pydantic AI traces
- Only simulation traces are nested under the parent run

In [5]:
from langgraph.types import Command

# Wrap simulation in session context for unified MLflow tracking
with simulation_session(config, scenario_id=scenario_id) as (session_id, graph_config):
    print(f"\n📋 Session ID: {session_id}")
    print(
        f"📊 Session Name: sim-{config.activity_type}-{config.num_participants}p-{config.difficulty}-...\n"
    )

    current_state = initial_state
    turn_count = 0
    simulation_complete = False

    print("\n=== STARTING SIMULATION ===\n")

    while not simulation_complete:
        turn_count += 1
        print(f"--- TURN {turn_count} ---")

        # Only pass state on first turn; subsequent turns use checkpointed state
        # Passing accumulated state causes duplication with the reducer pattern
        input_state = initial_state if turn_count == 1 else None
        async for event in graph.astream(input_state, graph_config):
            if "__interrupt__" in event:
                interrupt_obj = event["__interrupt__"]
                if hasattr(interrupt_obj, "value"):
                    interrupt_data = interrupt_obj.value
                elif isinstance(interrupt_obj, (list, tuple)):
                    interrupt_data = (
                        interrupt_obj[0].value
                        if hasattr(interrupt_obj[0], "value")
                        else interrupt_obj[0]
                    )
                else:
                    interrupt_data = interrupt_obj

                print(f"\n📖 {interrupt_data['narrative']}\n")
                print("Choices:")
                for i, choice in enumerate(interrupt_data["choices"], 1):
                    print(f"  {i}. {choice['description']}")

                while True:
                    try:
                        sel = int(input("\nSelect: ")) - 1
                        if 0 <= sel < len(interrupt_data["choices"]):
                            break
                        print("Invalid")
                    except ValueError:
                        print("Enter number")

                selected = interrupt_data["choices"][sel]
                current_state = await graph.ainvoke(
                    Command(resume={"choice_id": selected["choice_id"]}), graph_config
                )
                break

            # Check for graph completion
            if event == {}:
                simulation_complete = True
                break

        # Check if state indicates completion (AppState is TypedDict, use dict access)
        if current_state.get("is_complete"):
            simulation_complete = True

        # Safety limit
        if turn_count > 10:
            print("Safety stop")
            break

    print("\n=== SIMULATION COMPLETE ===")

    # Log final results to MLflow parent run
    log_simulation_results(
        transcript=current_state["transcript"],
        is_complete=current_state["is_complete"],
        key_learning_moments=current_state["key_learning_moments"],
    )
    print("\n✓ Results logged to MLflow")


📋 Session ID: 414ae229-fe3f-4d1c-8cdf-317522a92dea
📊 Session Name: sim-hiking-3p-med-...


=== STARTING SIMULATION ===

--- TURN 1 ---


2026/03/23 20:18:41 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f8439ecf6f0> at 0x7f83f555f600> was created in a different Context
2026/03/23 20:18:41 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f8439ecf6f0> at 0x7f83f556a780> was created in a different Context



📖 You encounter a group of three hikers. One man is sitting on a rock, hunched over, clutching his right side. His companions say he tripped 2 hours ago. He tells you his side hurts a lot, and he feels nauseous. What is your first priority?

Choices:
  1. Perform a quick physical check of his ankles and knees, then encourage him to walk slowly to the trailhead to get warm.
  2. Conduct a full vital sign check and a focused abdominal exam (asking about pain onset, location, and performing light palpation).


Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
2026/03/23 20:20:24 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f8439ecf6f0> at 0x7f83f578c380> was created in a different Context
2026/03/23 20:20:28 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f8439ecf6f0> at 0x7f83f56fe680> was created in a different Context
2026/03/23 20:20:28 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f8439ecf6f0> at 0x7f83f5568640> was created in a different Context
2026/03/23 20:20:28 WARNING mlflow.utils.autologging_utils: Encountered un

--- TURN 2 ---

📖 You perform the assessment. The patient is pale and tachycardic. He denies a direct blow to the abdomen. Upon light palpation, he guards the Right Lower Quadrant (RLQ). He explicitly winces when you lift your hand quickly after pressing on that area. How do you interpret these findings?

Choices:
  1. Assume a pulled muscle from the fall, administer over-the-counter painkillers, and advise a slow walk out.
  2. Note his elevated pulse and observe guarding in the RLQ, with pain exacerbating when you release pressure (rebound tenderness). Suspect a non-trauma emergency.


...

--- TURN 4 ---

📖 Your decision to initiate an urgent evacuation was correct. By recognizing the visceral nature of the pain and the classic presentation of acute appendicitis, you successfully prepared for a time-critical medical transfer, likely preventing a life-threatening rupture. The patient was safely transferred to SAR teams at the trailhead.

Choices:
  1. Scenario complete.
  2. Review performance.


2026/03/23 20:20:38 WARNING mlflow.tracing.fluent: Failed to start span LangGraph: 'NoneType' object has no attribute 'set_span_type'. For full traceback, set logging level to debug.
Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
Deserializing unregistered type summit_sim.schemas.ChoiceOption from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ChoiceOption')]
Deserializing unregistered type summit_sim.schemas.SimulationResult from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'SimulationResult')]
2026/03/23 20:20:38 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f8439ecf6f0> a


=== SIMULATION COMPLETE ===

✓ Results logged to MLflow
🏃 View run sim-noclass-hiking-3p-med at: https://mlflow.bhamm-lab.com/#/experiments/7/runs/bfdf488bcaf144fa9886ab4f86ecb6d4
🧪 View experiment at: https://mlflow.bhamm-lab.com/#/experiments/7


[Trace(trace_id=tr-bbff25a5594b7ba85d96234d4dc2f8db), Trace(trace_id=tr-0169bfa33dcc928a23f3c04269ddf569), Trace(trace_id=tr-4d4c60e54789b0dd8829628090936181), Trace(trace_id=tr-b9f1a002462f45ee1cac74604aea18ed)]

2026/03/23 20:20:42 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f8439ecf6f0> at 0x7f83f5448ec0> was created in a different Context


## 6. Display Results

Show the complete transcript and learning moments:

In [6]:
print("\n" + "=" * 60)
print("TRANSCRIPT")
print("=" * 60 + "\n")

for _i, entry in enumerate(current_state["transcript"], 1):
    print(f"Turn {entry['turn_id']}:")
    print(f"  Situation: {entry['turn_narrative'][:80]}...")
    print(f"  Choice: {entry['choice_description']}")
    print(f"  Feedback: {entry['feedback'][:100]}...")
    print(f"  Learning: {', '.join(entry['learning_moments'])}")
    print()


TRANSCRIPT

Turn 0:
  Situation: You encounter a group of three hikers. One man is sitting on a rock, hunched ove...
  Choice: Perform a quick physical check of his ankles and knees, then encourage him to walk slowly to the trailhead to get warm.
  Feedback: While it is natural to focus on the fall, you should not assume the injury is musculoskeletal just b...
  Learning: Always perform a comprehensive secondary assessment, encompassing more than just the site of suspected trauma, to avoid missing systemic distress., Systemic signs like tachycardia and nausea, paired with focal abdominal pain, are major indicators of a medical emergency that require prioritizing evacuation over self-extraction.

Turn 1:
  Situation: You perform the assessment. The patient is pale and tachycardic. He denies a dir...
  Choice: Note his elevated pulse and observe guarding in the RLQ, with pain exacerbating when you release pressure (rebound tenderness). Suspect a non-trauma emergency.
  Feedback: Excelle

In [7]:
print("\n" + "=" * 60)
print("KEY LEARNING MOMENTS")
print("=" * 60 + "\n")

for _i, moment in enumerate(current_state["key_learning_moments"], 1):
    print(f"{_i}. {moment}")


KEY LEARNING MOMENTS

1. Always perform a comprehensive secondary assessment, encompassing more than just the site of suspected trauma, to avoid missing systemic distress.
2. Systemic signs like tachycardia and nausea, paired with focal abdominal pain, are major indicators of a medical emergency that require prioritizing evacuation over self-extraction.
3. Rebound tenderness is a red-flag indicator of peritoneal irritation and should always raise suspicion for a serious internal, non-traumatic abdominal emergency like appendicitis.
4. Systemic signs such as tachycardia and pallor help distinguish between a localized musculoskeletal injury and a more serious condition affecting the patient’s overall status.
5. Deteriorating vital signs and systemic symptoms (fever, nausea) in the presence of abdominal pain are red flags for internal pathology rather than a simple musculoskeletal injury.
6. When you suspect a surgical emergency like appendicitis, the patient's best chance of survival is

## 7. MLflow Verification

Check your MLflow UI to verify:

### Trace Architecture
- **Scenario Generation** (Cell 2): Independent Pydantic AI trace
  - Tagged with `class_id` for linking to simulation
  - Appears as standalone trace in MLflow

- **Simulation** (Cell 5): Parent run with nested LangGraph traces
  - Parent Run: `sim-{class_id}-{activity}-{participants}p-{difficulty}`
  - Tagged with same `class_id` as generation
  - Session ID: UUID used for thread_id grouping
  - Child spans: Each agent call nested under parent

### Linking Generation and Simulation
Both phases share the same `class_id` tag:
- Filter in MLflow UI: `tags.class_id = "{class_id}"`
- Or search: `{class_id}` in run names
- This links: generation trace → simulation parent run

### Expected Behaviors
- **Interrupt 'Exceptions'**: Normal! LangGraph's `interrupt()` raises exceptions
  for human-in-the-loop. These appear in traces but are not errors.
- **Evaluation Tab**: Runs with traces appear here in MLflow 3.x UI
- **Session View**: Filter by thread_id to see grouped traces

### Logged Data
- **Metrics**: total_turns, is_complete, learning_moments_count
- **Tags**: class_id, session_type, activity_type, difficulty, status
- **Params**: class_id, session_id (UUID), activity_type, num_participants, difficulty

In [8]:
import mlflow

# Show experiment info
experiment = mlflow.get_experiment_by_name(settings.mlflow_experiment_name)
print(f"Experiment ID: {experiment.experiment_id}")
print(f"Artifact Location: {experiment.artifact_location}")
print(f"\nView traces at: {settings.mlflow_tracking_uri}")
print("\nIn MLflow UI, look for:")
print(f"  📁 class_id: {config.class_id}")
print(f"  📊 Generation trace: Filter by class_id tag or search '{config.class_id}'")
print(f"  📊 Simulation run: sim-{config.class_id}-... (parent run with nested traces)")
print("\nKey Features:")
print(f"  • class_id '{config.class_id}' links generation and simulation")
print('  • Filter: tags.class_id = "<your-class-id>"')
print("  • Metrics logged at session completion")
print("  • Status tag shows completed/failed")

Experiment ID: 7
Artifact Location: mlflow-artifacts:/7

View traces at: https://mlflow.bhamm-lab.com

In MLflow UI, look for:
  📁 class_id: None
  📊 Generation trace: Filter by class_id tag or search 'None'
  📊 Simulation run: sim-None-... (parent run with nested traces)

Key Features:
  • class_id 'None' links generation and simulation
  • Filter: tags.class_id = "<your-class-id>"
  • Metrics logged at session completion
  • Status tag shows completed/failed


---
## ✅ E2E Test Complete

If you successfully completed the simulation:
- ✓ LangGraph workflow executed correctly
- ✓ Human-in-the-loop interrupt worked
- ✓ AI feedback generated for each choice
- ✓ State advanced through all turns
- ✓ MLflow captured multi-step execution